# 加载pdf
PDF 存在多种来源格式，包括扫描版（图片 PDF）、电子文本版、混合版。并且布局格式也多种多样，包括单列布局、双列布局甚至竖排文本布局。并且包含段落、标题、页眉页脚、表格、数学公式、化学式、特殊符号、图片等各种元素。

因此，PDF 解析存在很多挑战。对于复杂 PDF，需要进行文本提取、布局检测、表格解析、公式识别等处理。

## 方式一：使用PyPDFLoader
### 需要安装pypdf： pip install pypdf
#### 1.简单pdf

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    file_path="../asset/load/04-sample.pdf",
    # 提取模式:控制如何从 PDF 文件中解析和提取文本结构。
    #   plain 提取文本，默认值
    #   layout 布局感知提取模式，通常会通过插入大量的空格、换行符，来模拟原文档中的多栏、缩进和间距（适用场景：学术论文（如 arXiv 论文）、多报刊杂志、带有左右分栏的合同）
    extraction_mode="plain"
)

docs = loader.load()
for doc in docs:
    print(doc)

page_content='中国科学院国家天文台 
2023 年部门预算' metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-07-24T17:46:07+08:00', 'title': '中国科学院国家天文台2023年度部门预算', 'author': 'MC SYSTEM', 'moddate': '2023-07-24T17:46:07+08:00', 'source': '../asset/load/04-sample.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1'}
page_content='目    录 
一、中国科学院国家天文台基本情况 ..................................... 1 
（一）单位职责 .................................................................... 1 
（二）机构设置 .................................................................... 2 
二、中国科学院国家天文台 2023 年部门预算 ....................... 3 
收支总表 ................................................................................ 4 
关于收支总表的说明 ............................................................ 5 
收入总表 ................................................................................ 6 
关于收入总表的说明 ............................................................ 7 
支出总表 ............................

#### 网站pdf
##### 需要安装 fontTools 库： pip install fonttools


In [1]:
from langchain_community.document_loaders import PyPDFLoader
import fontTools

loader = PyPDFLoader(
    # 文件路径，支持本地文件和在线文件链接
    # file_path="../asset/load/04-sample.pdf",
    file_path="https://arxiv.org/pdf/alg-geom/9202012",
    # 提取模式:控制如何从 PDF 文件中解析和提取文本结构。
    #   plain 提取文本，默认值
    #   layout 布局感知提取模式，通常会通过插入大量的空格、换行符，来模拟原文档中的多栏、缩进和间距（适用场景：学术论文（如 arXiv 论文）、多栏报刊杂志、带有左右分栏的合同）
    extraction_mode="plain",
)

docs = loader.load()

print(docs)
print(len(docs))



C:\Users\kenny\AppData\Local\Temp\ipykernel_18144\2875908078.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[Document(metadata={'producer': 'GPL Ghostscript GIT PRERELEASE 9.22', 'creator': 'dvips(k) 5.86 Copyright 1999 Radical Eye Software', 'creationdate': '2021-07-02T11:38:36-04:00', 'moddate': '2021-07-02T11:38:36-04:00', 'title': 'arXiv:alg-geom/9202012v1  14 Feb 1992', 'source': 'https://arxiv.org/pdf/alg-geom/9202012', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='arXiv:alg-geom/9202012v1  14 Feb 1992\nCohomology of coherent sheaves on a proper scheme\nGeorge R. Kempf\nThe Johns Hopkins University\nLet f : X → S be a proper morphism where S = Spec A where A is a noetherian\nring. Let F be a coherent sheaf on X. Then a theorem of Grothendieck says that the\ncohomology groups H i(X, F ) are A-modules of ﬁnite type. We set out to give an algorithm\nto compute these A-modules when A is suﬃciently algorithmic. We found general result\nabout a Γ( X, −)-acylic resolution of F .\nWe will ﬁrst deﬁne a coherent sheaf G on X to be primitive if support ( G) = ¯x where x\nis a poi

## 方式2.使用minerU
### 1.每90天需要申请新的token，网址： https://mineru.net/apiManage/token
### 2.需要安装requests库： pip install requests


In [2]:
import os
import time
import requests
from dotenv import load_dotenv

load_dotenv(verbose=True)

def upload_file(file_paths: list[str]) -> str:
    """批量上传文件"""
    url="https://mineru.net/api/v4/file-urls/batch"
    token = os.getenv("MINERU_API_TOKEN")
    ## 设置headers
    headers ={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }
    ## 设置文件信息
    file_info = [
        {
            "name": os.path.basename(file_path),
            "is_ocr":True,
            "data_id":f"file_{i}"
        } for i, file_path in enumerate(file_paths)
    ]

    ## 设置数据格式
    data = {
        "enable_formula":True,
        "enable_table":True,
        "language":"ch",
        "files":file_info
    }
    try:
      resp = requests.post(url, headers=headers, json=data)
      if resp.status_code == 200: #成功
          result = resp.json()
          print("response success. result:{}".format(result))
          if result["code"] == 0:
              batch_id = result["data"]["batch_id"]
              urls = result["data"]["file_urls"]
              print("batch_id:{}\nurls:{}".format(batch_id, urls))
              for i in range(len(urls)):
                  with open(file_paths[i], "rb") as f:
                      res_upload = requests.put(urls[i], data=f)
                      if res_upload.status_code == 200: # 更新成功
                          if res_upload.status_code == 200:
                            print(f"{urls[i]} upload success")
                          else:
                            print(f"{urls[i]} upload failed")
                            return "None"
              return batch_id
          else:
              print("apply upload url failed, reason:{}".format(result.get("msg")))
              return "None"
      else:
          print(
                "response not success. status:{} ,result:{}".format(
                    resp.status_code, resp.text
                )
            )
          return "None"
    except Exception as e:
        print(e)
        return  "None"

def download_files(batch_id):
    """批量获取任务结果"""
    # 如果batch_id为空，就跳过下载，直接返回
    if not batch_id:
        print("batch_id是空的，跳过下载")
        return

    # 创建一个保存结果的文件夹
    os.makedirs("parsed_files", exist_ok=True)
    url = f"https://mineru.net/api/v4/extract-results/batch/{batch_id}"
    token = os.getenv("MINERU_API_TOKEN")
    headers ={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

    failed_files = set()
    done_files = set()
    while True:
        res = requests.get(url, headers=headers)
        res_json = res.json()
        if res.status_code != 200 or res_json["code"] != 0:
            print(f"get result failed:{res_json}")
            break # 拿不到结果就退出
        extract_result = res_json["data"]["extract_result"]
        for result in extract_result:
            # 获取data_id
            data_id = result["data_id"]
            # 检查状态，如果失败，就把结果添加到失败列表
            if result["state"] == "failed":
                failed_files.add(data_id)
            # 如果成功，就把结果添加到成功列表,注意还有检查一下data_id是否语句存在，如果不存在才添加
            elif result["state"] == "done" and data_id not in done_files:
                done_files.add(data_id)
                # 获取他的zip文件url
                full_zip_url = result["full_zip_url"]
                # 获取这个文件的内容
                res_download = requests.get(full_zip_url, stream=True)
                # 保存文件内容
                with open(f"parsed_files/{result['file_name']}_{result['data_id']}.zip", "wb") as f:
                    for chunk in res_download.iter_content(chunk_size=1024):
                        if chunk:
                            f.write(chunk)
        #  failed_files的总长度+  done_files ==  extract_result的长度，说明下载完成
        if len(failed_files) + len(done_files) == len(extract_result):
            break
        # 预防爬取太快，把服务器高崩溃
        time.sleep(5)
        # 分别输出成功的文件回到失败的文件
        for i in failed_files:
            print(f"失败文件：{i}")
        for i in done_files:
            print(f"处理文件：{i}")

file_paths = ["../asset/load/04-sample.pdf"]
batch_id = upload_file(file_paths)
if batch_id:
    download_files(batch_id)









response success. result:{'code': 0, 'msg': 'ok', 'trace_id': 'ec26e71dfd7351a96f77c6080c469ca1', 'data': {'batch_id': '45290f20-d34e-403f-9643-9eb199dd01e0', 'file_urls': ['https://mineru.oss-cn-shanghai.aliyuncs.com/api-upload/extract/2026-11-07/45290f20-d34e-403f-9643-9eb199dd01e0/3ee3c3b4-a615-4b67-9c76-9e06f4b0bd83.pdf?Expires=1794073556&OSSAccessKeyId=LTAI5t6sXp5gszd8yWZ85dAL&Signature=PUcyTCznktCaa3chO4SW7WhVSAI%3D']}}
batch_id:45290f20-d34e-403f-9643-9eb199dd01e0
urls:['https://mineru.oss-cn-shanghai.aliyuncs.com/api-upload/extract/2026-11-07/45290f20-d34e-403f-9643-9eb199dd01e0/3ee3c3b4-a615-4b67-9c76-9e06f4b0bd83.pdf?Expires=1794073556&OSSAccessKeyId=LTAI5t6sXp5gszd8yWZ85dAL&Signature=PUcyTCznktCaa3chO4SW7WhVSAI%3D']
https://mineru.oss-cn-shanghai.aliyuncs.com/api-upload/extract/2026-11-07/45290f20-d34e-403f-9643-9eb199dd01e0/3ee3c3b4-a615-4b67-9c76-9e06f4b0bd83.pdf?Expires=1794073556&OSSAccessKeyId=LTAI5t6sXp5gszd8yWZ85dAL&Signature=PUcyTCznktCaa3chO4SW7WhVSAI%3D upload succ